# 질문셋들 테스트

할 수 있는 것:
- 질문 유형별 테스트셋 확인
- 전체 질문 일괄 실행
- 유형별 선택 실행
- 실제 Generation 호출 또는 프롬프트 미리보기(prompt only) 실행
- 결과를 `csv`/`json`으로 저장
- 응답 형식 / 출처 / 추정 여부 등을 수동 체크

권장 사용 순서:
1. 경로/모드 설정
2. 질문셋 확인
3. `RUN_MODE`를 `prompt_only` 또는 `generate`로 선택
4. 전체 또는 유형별 실행
5. 결과 확인 및 CSV 저장


In [17]:

from pathlib import Path
import sys
import os

cwd = Path.cwd().resolve()

candidate_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
]

project_root = None
for root in candidate_roots:
    if (root / "src").exists():
        project_root = root
        break

if project_root is None:
    project_root = cwd

src_path = project_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("project_root =", project_root)
print("src_path =", src_path)
print("src_exists =", src_path.exists())


project_root = C:\개발프로젝트\코드잇-4팀-중급프로젝트-Bidcoin\Bidcoin
src_path = C:\개발프로젝트\코드잇-4팀-중급프로젝트-Bidcoin\Bidcoin\src
src_exists = True


In [18]:

import json
from copy import deepcopy
from datetime import datetime

try:
    import pandas as pd
except ImportError:
    pd = None

from generation.mock_data import get_mock_retrieval_result
from generation.context_builder import build_context_block, build_history_block
from generation.prompts import SYSTEM_PROMPT, build_user_prompt
from generation.generator import BidCoinGenerator
from generation.schemas import RetrievalResult
from generation.config import Settings


## 1. 질문셋 정의

예상 질문 흐름을 바탕으로 만든 1차 Generation 테스트셋입니다.

각 카테고리 역할
- fact: 예산, 기관, 일정 같은 사실 추출
- condition: 가능 여부, 요구 여부, 자격 여부
- summary: 정리/요약
- compare: 비교
- recommend: 추천/판단
- evidence: 근거/원문 강조
- followup: 이전 맥락 이어받기
- refusal: 문서에 없는 질문 잘 거절하는지
- complex: 여러 요청이 섞인 난도 높은 질문


In [19]:

TEST_SETS = {
    "fact": [
        "발주기관이 어디야?",
        "공고명 뭐야?",
        "예산 얼마야?",
        "사업 기간은 어떻게 돼?",
        "제출 마감일은 언제야?",
        "필수 제출 서류가 뭐야?",
    ],
    "condition": [
        "입찰 참가 자격요건이 있어?",
        "공동수급 가능해?",
        "하도급 관련 조건이 있어?",
        "유지보수 요구가 포함돼 있어?",
        "보안 요구사항이 있어?",
        "개인정보 관련 요구사항이 있어?",
    ],
    "summary": [
        "이 사업의 목적과 배경을 5줄 이내로 정리해줘.",
        "주요 요구사항 3개만 알려줘.",
        "핵심 기능만 요약해줘.",
        "기능 요구사항과 비기능 요구사항을 나눠서 정리해줘.",
        "구축 범위와 운영 범위를 구분해서 설명해줘.",
        "입찰 참여 여부 판단에 필요한 핵심 정보만 요약해줘.",
    ],
    "compare": [
        "고려대학교 사업이랑 광주과학기술원 사업을 비교해줘.",
        "두 사업의 목적 차이를 정리해줘.",
        "예산 규모가 더 큰 쪽은 어디야?",
        "일정이 더 촉박한 사업은 뭐야?",
        "기능 요구사항이 더 많은 쪽은 어디야?",
        "두 사업 중 운영 부담이 더 큰 쪽은 어디라고 보이는지 문서 근거로 비교해줘.",
    ],
    "recommend": [
        "우리 고객사가 대학 시스템 구축 경험이 많은데, 이 사업이 적합한지 판단해줘.",
        "교육 플랫폼 구축 경험이 많은 회사에 추천할 만한 공고인지 설명해줘.",
        "유지보수 조직이 강한 회사에 유리한 사업인지 판단해줘.",
        "이 사업이 중소기업도 도전 가능한 공고인지 문서 근거로 판단해줘.",
        "이 사업의 진입장벽이 높다고 볼 수 있는지 정리해줘.",
        "마지막에 go / no-go 의견을 문서 근거만 바탕으로 제한적으로 제시해줘.",
    ],
    "evidence": [
        "문서 근거랑 같이 보여줘.",
        "보안 요구사항의 근거 문장을 보여줘.",
        "제출 방식 관련 원문 표현을 최대한 유지해서 알려줘.",
        "방금 답변의 근거가 된 문장만 다시 보여줘.",
        "확실한 내용만 남기고 근거와 함께 다시 정리해줘.",
        "원문 기준으로 필수 제출 서류를 보여줘.",
    ],
    "followup": [
        "그중 필수 조건만 다시 정리해줘.",
        "아니, 선택 조건 말고 반드시 해야 하는 것만.",
        "예산 말고 일정 중심으로 다시 설명해줘.",
        "아까 비교한 두 사업 중 응답속도 요구만 다시 봐줘.",
        "이번엔 고객사에게 보낼 수 있게 더 짧게 써줘.",
        "이번엔 컨설턴트 내부 메모 스타일로 바꿔줘.",
    ],
    "refusal": [
        "삼성전자 관련 RFP도 있어?",
        "이 사업에 AWS 사용이 명시돼 있어?",
        "이 문서에 없는 기술 요구까지 포함해서 추정해서 설명해줘.",
        "발주기관의 실제 내부 의도를 분석해줘.",
        "경쟁사들이 어떤 제안을 낼지 예측해줘.",
        "문서에 없는 수주 가능성 수치를 계산해줘.",
    ],
    "complex": [
        "이 사업의 목적, 주요 기능 요구사항, 운영 및 유지보수 요구, 제출 방식, 평가 요소를 항목별로 정리하고 마지막에 어떤 고객사에게 적합한지 제한적으로 판단해줘.",
        "여러 장에 흩어진 일정, 제출 요건, 인력 요구, 기능 요구, 보안 요구를 통합해서 컨설턴트 내부 검토 메모 형식으로 재구성해줘.",
        "이 공고가 우리 고객사에게 실질적으로 추천 가능한지 판단하기 위해, 참여 자격, 기술 적합성, 수행 난이도, 일정 리스크를 각각 정리해줘.",
        "비교한 두 사업 중 어떤 유형의 SI 업체에 각각 더 적합한지 문서 근거만 바탕으로 설명해줘.",
        "이 사업에서 놓치기 쉬운 조건이 있는지 체크리스트 형태로 정리해줘.",
        "근거 없는 추정은 하지 말고, 불확실한 항목은 따로 표시해서 다시 답해줘.",
    ],
}


In [20]:

def flatten_test_sets(test_sets: dict) -> list[dict]:
    rows = []
    for category, questions in test_sets.items():
        for i, q in enumerate(questions, start=1):
            rows.append({
                "category": category,
                "question_no": i,
                "question": q,
            })
    return rows

test_rows = flatten_test_sets(TEST_SETS)
print(f"총 질문 수: {len(test_rows)}")

if pd is not None:
    test_df = pd.DataFrame(test_rows)
    display(test_df)
else:
    test_rows[:5]


총 질문 수: 54


## 2. 실행 모드 설정

### `RUN_MODE`
- `prompt_only`: OpenAI 호출 없이 **프롬프트만 미리보기**
- `generate`: 실제 모델 호출

### `INPUT_MODE`
- `mock`: `mock_data.py`의 RetrievalResult 사용
- `json`: Retrieval 단계에서 넘어온 JSON 파일을 불러와 사용

> 일단`prompt_only + mock` 조합으로 진행


In [21]:

# ===== 사용자 설정 =====
RUN_MODE = "generate"   # "prompt_only" or "generate"
INPUT_MODE = "mock"        # "mock" or "json"

# INPUT_MODE == "json" 일 때 사용할 파일 경로
RETRIEVAL_JSON_PATH = ""

# 실행할 유형 선택
# 예: ["fact", "summary"] / None 이면 전체 실행
SELECTED_CATEGORIES = None

# 각 유형에서 몇 개씩만 돌릴지 제한하고 싶다면 숫자 지정
# None 이면 해당 유형 전체
MAX_PER_CATEGORY = None

# 실제 generation 호출 시 결과를 너무 길게 출력하지 않기 위한 preview 길이
ANSWER_PREVIEW_CHARS = 500


## 3. Retrieval 입력 로드 유틸리티

In [22]:

def load_retrieval_result(input_mode: str = "mock", json_path: str = ""):
    if input_mode == "mock":
        return get_mock_retrieval_result()

    if input_mode == "json":
        if not json_path:
            raise ValueError("INPUT_MODE='json'이면 RETRIEVAL_JSON_PATH를 지정해야 합니다.")

        path = Path(json_path)
        if not path.exists():
            raise FileNotFoundError(f"JSON 파일을 찾을 수 없습니다: {path}")

        data = json.loads(path.read_text(encoding="utf-8"))
        return RetrievalResult.model_validate(data)

    raise ValueError("input_mode는 'mock' 또는 'json' 이어야 합니다.")


In [23]:

base_result = load_retrieval_result(INPUT_MODE, RETRIEVAL_JSON_PATH)
base_result


RetrievalResult(question='콘텐츠 관리 요구사항과 보안 요구사항을 정리해줘.', contexts=[RetrievedContext(chunk_id='doc_001_chunk_01', text='본 사업은 이러닝 시스템 기능 고도화를 목적으로 한다. 학습 콘텐츠 등록, 수정, 버전관리 기능을 제공해야 하며, 관리자는 학습 이력과 진도 현황을 조회할 수 있어야 한다.', source_file='국민연금공단_이러닝시스템.hwp', organization='국민연금공단', project_name='이러닝시스템 구축', summary='이러닝 콘텐츠 관리 및 학습 이력 조회 기능을 포함한 시스템 고도화 사업', score=0.93), RetrievedContext(chunk_id='doc_001_chunk_02', text='보안 요구사항으로는 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능이 요구된다.', source_file='국민연금공단_이러닝시스템.hwp', organization='국민연금공단', project_name='이러닝시스템 구축', summary='보안 및 개인정보보호 요구사항 포함', score=0.89), RetrievedContext(chunk_id='doc_001_chunk_03', text='운영 측면에서는 장애 대응 체계, 백업 및 복구 방안, 운영 매뉴얼 제공이 요구된다.', source_file='국민연금공단_이러닝시스템.hwp', organization='국민연금공단', project_name='이러닝시스템 구축', summary='운영 및 유지보수 요구사항 포함', score=0.84)], chat_history=[ChatTurn(role='user', content='국민연금공단 사업 문서를 찾아줘.'), ChatTurn(role='assistant', content='국민연금공단 이러닝시스템 구축 관련 문서를 참고하겠습니다.')])

## 4. Prompt 미리보기

실제 모델 호출 전에, 모델에 어떤 문자열이 들어가는지부터 확인


In [24]:

def make_prompt_preview(retrieval_result):
    settings = Settings()
    context_block, used_sources = build_context_block(
        contexts=retrieval_result.contexts,
        max_contexts=settings.max_contexts,
        max_chars=settings.max_context_chars,
    )
    history_block = build_history_block(
        retrieval_result.chat_history,
        max_turns=3
    )
    user_prompt = build_user_prompt(
        question=retrieval_result.question,
        context_block=context_block,
        history_block=history_block,
    )
    return {
        "context_block": context_block,
        "history_block": history_block,
        "user_prompt": user_prompt,
        "used_sources": used_sources,
    }

preview = make_prompt_preview(base_result)
print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print("\n=== USER PROMPT ===")
print(preview["user_prompt"])
print("\n=== USED SOURCES ===")
print(preview["used_sources"])


=== SYSTEM PROMPT ===
당신은 B2G 공공입찰 전문 컨설팅 어시스턴트입니다.
반드시 주어진 RFP 문서 컨텍스트만을 근거로만 답변하세요.

공통 규칙:
1. 문서에 없는 내용은 추측하지 말고 "해당 문서에서 확인할 수 없습니다."라고 답하세요.
2. 답변은 한국어로 작성하세요.
3. 문서 근거가 있는 내용만 답하세요.
4. 마지막에 반드시 "출처" 섹션을 넣고 파일명을 나열하세요.
5. 사용자가 판단/추천을 요청하더라도 문서에 드러난 근거 범위 안에서만 제한적으로 설명하세요.
6. 불확실한 내용은 "불확실"로 표시하세요.

=== USER PROMPT ===
[참고 문서]
[문서 1] chunk_id=doc_001_chunk_01 | score=0.9300 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 이러닝 콘텐츠 관리 및 학습 이력 조회 기능을 포함한 시스템 고도화 사업
본문:
본 사업은 이러닝 시스템 기능 고도화를 목적으로 한다. 학습 콘텐츠 등록, 수정, 버전관리 기능을 제공해야 하며, 관리자는 학습 이력과 진도 현황을 조회할 수 있어야 한다.

---

[문서 2] chunk_id=doc_001_chunk_02 | score=0.8900 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 보안 및 개인정보보호 요구사항 포함
본문:
보안 요구사항으로는 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능이 요구된다.

---

[문서 3] chunk_id=doc_001_chunk_03 | score=0.8400 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 운영 및 유지보수 요구사항 포함
본문:
운영 측면에서는 장애 대응 체계, 백업 및 복구 방안, 운영 매뉴얼 제공이 요구된다.

[최근 대화]
- 사용자: 국민연금공단 사업 문서를 찾아줘.
- 어시스

## 5. 단일 질문 실행 함수

이 함수는 질문 1개를 실행합니다.
- `prompt_only` 모드면 프롬프트만 생성
- `generate` 모드면 실제 모델 호출


In [25]:

def clone_retrieval_result_with_question(base_retrieval_result, question: str):
    copied = deepcopy(base_retrieval_result)
    copied.question = question
    return copied


def run_single_question(question: str, base_retrieval_result, run_mode: str = "prompt_only") -> dict:
    rr = clone_retrieval_result_with_question(base_retrieval_result, question)

    prompt_preview = make_prompt_preview(rr)

    result = {
        "question": question,
        "used_sources": prompt_preview["used_sources"],
        "status": "ok",
        "mode": run_mode,
        "answer": None,
        "answer_preview": None,
        "context_preview": prompt_preview["context_block"][:800],
        "user_prompt_preview": prompt_preview["user_prompt"][:2000],
        "error": None,
    }

    if run_mode == "prompt_only":
        result["answer"] = "[PROMPT ONLY MODE] 실제 모델 호출 없이 프롬프트만 생성했습니다."
        result["answer_preview"] = result["answer"]
        return result

    if run_mode == "generate":
        try:
            generator = BidCoinGenerator()
            response = generator.generate(rr)
            result["answer"] = response.answer
            result["answer_preview"] = response.answer[:ANSWER_PREVIEW_CHARS]
            result["used_context_count"] = response.used_context_count
            result["raw_model_output"] = response.raw_model_output
            return result
        except Exception as e:
            result["status"] = "error"
            result["error"] = str(e)
            return result

    raise ValueError("run_mode는 'prompt_only' 또는 'generate' 이어야 합니다.")


## 6. 단일 질문 수동 테스트

In [26]:

sample_question = TEST_SETS["fact"][0]  # 필요하면 바꿔도 됨
single_result = run_single_question(sample_question, base_result, RUN_MODE)

print("=== QUESTION ===")
print(single_result["question"])
print("\n=== STATUS ===")
print(single_result["status"])
print("\n=== ANSWER PREVIEW ===")
print(single_result["answer_preview"])
print("\n=== USED SOURCES ===")
print(single_result["used_sources"])
print("\n=== USER PROMPT PREVIEW ===")
print(single_result["user_prompt_preview"])


=== QUESTION ===
발주기관이 어디야?

=== STATUS ===
ok

=== ANSWER PREVIEW ===
1. 답변
발주기관은 국민연금공단입니다.

2. 근거
- 문서 메타정보에 기관으로 '국민연금공단'이 표기되어 있습니다.
- 문서 요약 및 본문에서 '국민연금공단 이러닝시스템 구축' 사업으로 명시되어 있습니다.

3. 출처
국민연금공단_이러닝시스템.hwp

=== USED SOURCES ===
['국민연금공단_이러닝시스템.hwp']

=== USER PROMPT PREVIEW ===
[참고 문서]
[문서 1] chunk_id=doc_001_chunk_01 | score=0.9300 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 이러닝 콘텐츠 관리 및 학습 이력 조회 기능을 포함한 시스템 고도화 사업
본문:
본 사업은 이러닝 시스템 기능 고도화를 목적으로 한다. 학습 콘텐츠 등록, 수정, 버전관리 기능을 제공해야 하며, 관리자는 학습 이력과 진도 현황을 조회할 수 있어야 한다.

---

[문서 2] chunk_id=doc_001_chunk_02 | score=0.8900 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 보안 및 개인정보보호 요구사항 포함
본문:
보안 요구사항으로는 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능이 요구된다.

---

[문서 3] chunk_id=doc_001_chunk_03 | score=0.8400 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 운영 및 유지보수 요구사항 포함
본문:
운영 측면에서는 장애 대응 체계, 백업 및 복구 방안, 운영 매뉴얼 제공이 요구된다.

[최근 대화]
- 사용자: 국민연금공단 사업 문서를 찾아줘.
- 어시스턴트: 국민연금공단 이러닝시스템 구축 관련 문서를 참고하겠습니다.

[

## 7. 배치 실행 함수

- 전체 질문 일괄 실행
- 유형별 선택 실행
- 유형별 개수 제한 실행


In [27]:

def select_questions(test_sets, selected_categories=None, max_per_category=None):
    selected = []
    categories = selected_categories or list(test_sets.keys())

    for category in categories:
        questions = test_sets[category]
        if max_per_category is not None:
            questions = questions[:max_per_category]

        for i, q in enumerate(questions, start=1):
            selected.append({
                "category": category,
                "question_no": i,
                "question": q,
            })
    return selected


def run_batch(test_sets, base_retrieval_result, run_mode="prompt_only",
              selected_categories=None, max_per_category=None):
    selected_rows = select_questions(
        test_sets=test_sets,
        selected_categories=selected_categories,
        max_per_category=max_per_category,
    )

    outputs = []
    total = len(selected_rows)

    for idx, row in enumerate(selected_rows, start=1):
        print(f"[{idx}/{total}] {row['category']} - {row['question_no']}: {row['question']}")
        out = run_single_question(
            question=row["question"],
            base_retrieval_result=base_retrieval_result,
            run_mode=run_mode,
        )
        out["category"] = row["category"]
        out["question_no"] = row["question_no"]
        outputs.append(out)

    return outputs


## 8. 전체 또는 유형별 일괄 실행

이 셀을 실행하면 질문셋을 한 번에 돌립니다.
- `RUN_MODE='prompt_only'`면 빠르게 전체 프롬프트 구조 점검
- `RUN_MODE='generate'`면 실제 답변 생성


In [28]:

batch_results = run_batch(
    test_sets=TEST_SETS,
    base_retrieval_result=base_result,
    run_mode=RUN_MODE,
    selected_categories=SELECTED_CATEGORIES,
    max_per_category=MAX_PER_CATEGORY,
)

print(f"총 실행 결과 수: {len(batch_results)}")


[1/54] fact - 1: 발주기관이 어디야?
[2/54] fact - 2: 공고명 뭐야?
[3/54] fact - 3: 예산 얼마야?
[4/54] fact - 4: 사업 기간은 어떻게 돼?
[5/54] fact - 5: 제출 마감일은 언제야?
[6/54] fact - 6: 필수 제출 서류가 뭐야?
[7/54] condition - 1: 입찰 참가 자격요건이 있어?
[8/54] condition - 2: 공동수급 가능해?
[9/54] condition - 3: 하도급 관련 조건이 있어?
[10/54] condition - 4: 유지보수 요구가 포함돼 있어?
[11/54] condition - 5: 보안 요구사항이 있어?
[12/54] condition - 6: 개인정보 관련 요구사항이 있어?
[13/54] summary - 1: 이 사업의 목적과 배경을 5줄 이내로 정리해줘.
[14/54] summary - 2: 주요 요구사항 3개만 알려줘.
[15/54] summary - 3: 핵심 기능만 요약해줘.
[16/54] summary - 4: 기능 요구사항과 비기능 요구사항을 나눠서 정리해줘.
[17/54] summary - 5: 구축 범위와 운영 범위를 구분해서 설명해줘.
[18/54] summary - 6: 입찰 참여 여부 판단에 필요한 핵심 정보만 요약해줘.
[19/54] compare - 1: 고려대학교 사업이랑 광주과학기술원 사업을 비교해줘.
[20/54] compare - 2: 두 사업의 목적 차이를 정리해줘.
[21/54] compare - 3: 예산 규모가 더 큰 쪽은 어디야?
[22/54] compare - 4: 일정이 더 촉박한 사업은 뭐야?
[23/54] compare - 5: 기능 요구사항이 더 많은 쪽은 어디야?
[24/54] compare - 6: 두 사업 중 운영 부담이 더 큰 쪽은 어디라고 보이는지 문서 근거로 비교해줘.
[25/54] recommend - 1: 우리 고객사가 대학 시스템 구축 경험이 많은데, 이 사업이 적합한지 판

In [29]:

if pd is not None:
    results_df = pd.DataFrame(batch_results)
    display(
        results_df[
            [
                "category",
                "question_no",
                "question",
                "status",
                "answer_preview",
                "used_sources",
                "error",
            ]
        ]
    )
else:
    batch_results[:3]


## 9. 결과 저장

- batch 결과: 실행 결과 전체
- manual eval: 사람이 체크한 평가표


In [31]:

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

batch_json_path = output_dir / f"generation_batch_results_{timestamp}.json"
batch_json_path.write_text(
    json.dumps(batch_results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("saved:", batch_json_path)

if pd is not None:
    batch_csv_path = output_dir / f"generation_batch_results_{timestamp}.csv"
    results_df.to_csv(batch_csv_path, index=False, encoding="utf-8-sig")
    print("saved:", batch_csv_path)

    eval_csv_path = output_dir / f"generation_manual_eval_{timestamp}.csv"
    manual_eval_df.to_csv(eval_csv_path, index=False, encoding="utf-8-sig")
    print("saved:", eval_csv_path)


saved: C:\개발프로젝트\코드잇-4팀-중급프로젝트-Bidcoin\Bidcoin\outputs\generation_batch_results_20260409_155728.json


## 10. 사용 시나리오

### A. 프롬프트만 빠르게 확인하고 싶을 때
- `RUN_MODE = "prompt_only"`
- `INPUT_MODE = "mock"` 또는 `json`

### B. 실제 Retrieval JSON으로 end-to-end 테스트할 때
- `RUN_MODE = "generate"`
- `INPUT_MODE = "json"`
- `RETRIEVAL_JSON_PATH` 지정

### C. 유형별로만 돌리고 싶을 때
예:
```python
SELECTED_CATEGORIES = ["fact", "condition", "summary"]
```

### D. 각 유형당 2개씩만 먼저 보고 싶을 때
```python
MAX_PER_CATEGORY = 2
```
